# Lower Third Generator
Face detection & automatic lower third placement for broadcast post-production.

**Pipeline:** Upload proxy video → Detect faces → Cluster per person → Label → Download CSV / AVID / ProRes 4444

## 1. Install dependencies

In [ ]:
!pip install -q insightface onnxruntime opencv-python-headless scikit-learn numpy
print('Dependencies installed.')

## 2. Upload your proxy video

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
VIDEO_PATH = list(uploaded.keys())[0]
print(f'Uploaded: {VIDEO_PATH} ({os.path.getsize(VIDEO_PATH) / 1024 / 1024:.1f} MB)')

## 3. Settings
Adjust these before running detection.

In [ ]:
#@title Settings { run: "auto" }

SAMPLE_INTERVAL = 12       #@param {type:"integer"}
DETECTION_THRESHOLD = 0.5  #@param {type:"number"}
CLUSTER_EPS = 0.65         #@param {type:"number"}
LOWER_THIRD_DURATION = 5.0 #@param {type:"number"}
RENDER_PRORES = True       #@param {type:"boolean"}
EXPORT_AVID = True         #@param {type:"boolean"}

print(f'Sample every {SAMPLE_INTERVAL} frames | Threshold: {DETECTION_THRESHOLD}')
print(f'Cluster eps: {CLUSTER_EPS} | Duration: {LOWER_THIRD_DURATION}s')
print(f'Render ProRes: {RENDER_PRORES} | AVID export: {EXPORT_AVID}')

## 4. Detect faces & cluster

In [ ]:
import csv
import subprocess
from dataclasses import dataclass, field

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from insightface.app import FaceAnalysis
from sklearn.cluster import DBSCAN


@dataclass
class FaceDetection:
    frame_number: int
    embedding: np.ndarray
    bbox: tuple
    confidence: float


@dataclass
class PersonCluster:
    cluster_id: int
    detections: list = field(default_factory=list)
    name: str = ""
    title: str = ""

    @property
    def frame_numbers(self):
        return sorted(set(d.frame_number for d in self.detections))


@dataclass
class TimecodeSegment:
    name: str
    title: str
    tc_in: str
    tc_out: str
    frame_in: int = 0
    frame_out: int = 0


def frame_to_timecode(frame_number, fps):
    total_frames = int(frame_number)
    ff = total_frames % round(fps)
    total_seconds = total_frames // round(fps)
    ss = total_seconds % 60
    total_minutes = total_seconds // 60
    mm = total_minutes % 60
    hh = total_minutes // 60
    return f'{hh:02d}:{mm:02d}:{ss:02d}:{ff:02d}'


# Initialize InsightFace
print('Loading InsightFace model...')
face_app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))

# Open video
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f'Video: {w}x{h} @ {fps:.2f}fps | {total_frames} frames | {total_frames/fps:.1f}s')

# Detect faces
print('\nDetecting faces...')
detections = []
frame_number = 0
sampled = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_number % SAMPLE_INTERVAL == 0:
        faces = face_app.get(frame)
        for face in faces:
            if face.det_score < DETECTION_THRESHOLD or face.embedding is None:
                continue
            detections.append(FaceDetection(
                frame_number=frame_number,
                embedding=face.embedding,
                bbox=tuple(face.bbox.astype(int)),
                confidence=float(face.det_score),
            ))
        sampled += 1
        if sampled % 50 == 0:
            pct = frame_number / total_frames * 100
            print(f'  {pct:.0f}% — {len(detections)} faces found')
    frame_number += 1

cap.release()
print(f'\nDone: {sampled} frames sampled, {len(detections)} face detections.')

# Cluster
print('\nClustering faces...')
embeddings = np.array([d.embedding for d in detections])
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1
embeddings_norm = embeddings / norms

labels = DBSCAN(eps=CLUSTER_EPS, min_samples=2, metric='cosine').fit_predict(embeddings_norm)
unique_labels = sorted(set(labels) - {-1})

clusters = []
for label in unique_labels:
    cluster = PersonCluster(cluster_id=label)
    for i, det in enumerate(detections):
        if labels[i] == label:
            cluster.detections.append(det)
    clusters.append(cluster)

noise = sum(1 for l in labels if l == -1)
print(f'Found {len(clusters)} person(s) ({noise} noise detections discarded)')
for c in clusters:
    frames = c.frame_numbers
    print(f'  Person {c.cluster_id}: {len(c.detections)} detections | {frame_to_timecode(frames[0], fps)} – {frame_to_timecode(frames[-1], fps)}')

## 5. Preview detected faces
Shows one sample frame per person so you know who to label.

In [ ]:
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(VIDEO_PATH)

fig, axes = plt.subplots(1, len(clusters), figsize=(5 * len(clusters), 5))
if len(clusters) == 1:
    axes = [axes]

for i, cluster in enumerate(clusters):
    # Pick a detection from the middle of the cluster
    mid_det = cluster.detections[len(cluster.detections) // 2]
    cap.set(cv2.CAP_PROP_POS_FRAMES, mid_det.frame_number)
    ret, frame = cap.read()
    if ret:
        x1, y1, x2, y2 = mid_det.bbox
        # Add some padding
        pad = 40
        y1 = max(0, y1 - pad)
        x1 = max(0, x1 - pad)
        y2 = min(frame.shape[0], y2 + pad)
        x2 = min(frame.shape[1], x2 + pad)
        face_crop = cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)
        axes[i].imshow(face_crop)
    axes[i].set_title(f'Person {cluster.cluster_id}\n{len(cluster.detections)} detections')
    axes[i].axis('off')

cap.release()
plt.tight_layout()
plt.show()

## 6. Label each person
Enter name and title/function for each detected person.

In [ ]:
for cluster in clusters:
    frames = cluster.frame_numbers
    tc_first = frame_to_timecode(frames[0], fps)
    tc_last = frame_to_timecode(frames[-1], fps)
    print(f'\nPerson {cluster.cluster_id} ({len(cluster.detections)} detections, {tc_first} – {tc_last})')
    cluster.name = input('  Name:  ').strip() or f'Person_{cluster.cluster_id}'
    cluster.title = input('  Title: ').strip()

print('\nLabeled:')
for c in clusters:
    print(f'  {c.name} — {c.title}')

## 7. Generate outputs
Builds timecode segments and writes all output files.

In [ ]:
os.makedirs('output', exist_ok=True)
base = os.path.splitext(VIDEO_PATH)[0]

# Build segments
segments = []
max_gap = SAMPLE_INTERVAL * 3

for cluster in clusters:
    if not cluster.name:
        continue
    frames = cluster.frame_numbers
    if not frames:
        continue

    seg_groups = []
    current = [frames[0]]
    for i in range(1, len(frames)):
        if frames[i] - frames[i-1] <= max_gap:
            current.append(frames[i])
        else:
            seg_groups.append(current)
            current = [frames[i]]
    seg_groups.append(current)

    for seg_frames in seg_groups:
        tc_in_frame = max(seg_frames[0] - 1, 0)
        lt_duration_frames = int(LOWER_THIRD_DURATION * fps)
        seg_end_frame = seg_frames[-1] + 1
        tc_out_frame = min(tc_in_frame + lt_duration_frames, seg_end_frame)

        segments.append(TimecodeSegment(
            name=cluster.name,
            title=cluster.title,
            tc_in=frame_to_timecode(tc_in_frame, fps),
            tc_out=frame_to_timecode(tc_out_frame, fps),
            frame_in=tc_in_frame,
            frame_out=tc_out_frame,
        ))

segments.sort(key=lambda s: s.tc_in)

# Write CSV
csv_path = f'output/{base}_lower_thirds.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['name', 'title', 'tc_in', 'tc_out'])
    for seg in segments:
        writer.writerow([seg.name, seg.title, seg.tc_in, seg.tc_out])
print(f'CSV: {csv_path}')

output_files = [csv_path]

# AVID SubCap
if EXPORT_AVID:
    subcap_path = f'output/{base}_lower_thirds.subcap.txt'
    with open(subcap_path, 'w') as f:
        f.write(f'@ Generated from {VIDEO_PATH}\n\n<begin subtitles>\n')
        for seg in segments:
            f.write(f'{seg.tc_in} {seg.tc_out}\n{seg.name}\n')
            if seg.title:
                f.write(f'{seg.title}\n')
            f.write('\n')
        f.write('<end subtitles>\n')
    print(f'AVID SubCap: {subcap_path}')
    output_files.append(subcap_path)

    marker_path = f'output/{base}_lower_thirds.markers.txt'
    with open(marker_path, 'w') as f:
        f.write('Color\tName\tComment\tTC1\tTC2\n')
        for seg in segments:
            comment = f'{seg.name} | {seg.title}' if seg.title else seg.name
            f.write(f'Cyan\t{seg.name}\t{comment}\t{seg.tc_in}\t{seg.tc_out}\n')
    print(f'AVID Markers: {marker_path}')
    output_files.append(marker_path)

print(f'\n{len(segments)} lower third(s):')
for seg in segments:
    print(f'  {seg.name}, {seg.title}, {seg.tc_in}, {seg.tc_out}')

## 8. Render ProRes 4444 with alpha (optional)
Generates a transparent .mov you can overlay in Premiere/AVID.

In [ ]:
if RENDER_PRORES:
    render_path = f'output/{base}_lower_thirds.mov'

    cap = cv2.VideoCapture(VIDEO_PATH)
    margin_x = int(w * 0.10)
    margin_y = int(h * 0.10)
    name_font_size = max(int(h * 0.035), 16)
    title_font_size = max(int(h * 0.025), 12)

    # Load fonts
    try:
        name_font = ImageFont.truetype('/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf', name_font_size)
        title_font = ImageFont.truetype('/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf', title_font_size)
    except:
        try:
            name_font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', name_font_size)
            title_font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', title_font_size)
        except:
            name_font = ImageFont.load_default()
            title_font = ImageFont.load_default()

    ffmpeg_cmd = [
        'ffmpeg', '-y', '-f', 'rawvideo', '-pix_fmt', 'rgba',
        '-s', f'{w}x{h}', '-r', str(fps), '-i', '-',
        '-c:v', 'prores_ks', '-profile:v', '4444',
        '-pix_fmt', 'yuva444p10le', render_path,
    ]
    ffmpeg_proc = subprocess.Popen(
        ffmpeg_cmd, stdin=subprocess.PIPE,
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE,
    )

    frame_number = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        active = [s for s in segments if s.frame_in <= frame_number <= s.frame_out]
        pil_img = Image.new('RGBA', (w, h), (0, 0, 0, 0))

        if active:
            draw = ImageDraw.Draw(pil_img, 'RGBA')
            for i, seg in enumerate(active):
                line_height = name_font_size + title_font_size + int(h * 0.015)
                text_y = h - margin_y - line_height - (i * (line_height + int(h * 0.02)))
                text_x = margin_x

                name_bbox = draw.textbbox((0, 0), seg.name, font=name_font)
                title_bbox = draw.textbbox((0, 0), seg.title, font=title_font)
                bar_w = max(name_bbox[2] - name_bbox[0], title_bbox[2] - title_bbox[0]) + int(w * 0.04)
                bar_h = line_height + int(h * 0.015)
                bar_x = text_x - int(w * 0.015)
                bar_y = text_y - int(h * 0.008)

                draw.rectangle([bar_x, bar_y, bar_x + bar_w, bar_y + bar_h], fill=(0, 0, 0, 160))
                draw.text((text_x, text_y), seg.name, font=name_font, fill=(255, 255, 255, 255))
                title_y = text_y + name_font_size + int(h * 0.005)
                draw.text((text_x, title_y), seg.title, font=title_font, fill=(200, 200, 200, 255))

        ffmpeg_proc.stdin.write(np.array(pil_img).tobytes())
        frame_number += 1

        if frame_number % 250 == 0:
            print(f'  Rendering: {frame_number / total_frames * 100:.0f}%')

    cap.release()
    ffmpeg_proc.stdin.close()
    ffmpeg_proc.wait()

    print(f'Rendered: {render_path} ({os.path.getsize(render_path) / 1024 / 1024:.1f} MB)')
    output_files.append(render_path)
else:
    print('ProRes render skipped (set RENDER_PRORES = True to enable)')

## 9. Download all outputs

In [ ]:
from google.colab import files as colab_files

print('Downloading files...')
for f in output_files:
    print(f'  {f}')
    colab_files.download(f)

print('\nDone!')